# SmartAgro Scoring — Инференс

Загрузка готовой модели + скоринг одного заявителя.

```
1. Загрузка модели (xgb_scorer.joblib)
2. Ввод данных заявителя
3. Предсказание (ML Score 0-100)
4. SHAP объяснение
```

In [ ]:
# ============================================================
# 1. Загрузка модели
# ============================================================
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

ML_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count",
    "debt_load_ratio", "log_amount", "livestock_count",
    "direction_code", "is_pedigree", "is_producer",
    "hour_submitted", "month_submitted", "region_encoded",
]

model = joblib.load("../models/xgb_scorer.joblib")
scaler = joblib.load("../models/scaler.joblib")

print("Модель загруена")
print(f"Фичей: {len(ML_FEATURES)}")

In [ ]:
# ============================================================
# 2. Ввод данных заявителя
# ============================================================

# Заполни свои значения:
applicant = {
    "gross_output_growth_yoy":      0.15,       # Рост валовой продукции (-0.3 до 0.8)
    "land_to_livestock_ratio":      3.0,        # Обеспеченность пастбищами Га/голову (0.2-10)
    "historical_survival_rate":     0.88,       # Сохранность поголовья (0.5-0.99)
    "subsidy_dependence_index":     0.35,       # Зависимость от субсидий (0-1)
    "veterinary_compliance":        0.85,       # Ветеринарное соответствие (0-1)
    "years_in_operation":           10.0,       # Лет работы (1-25)
    "pedigree_ratio":               0.60,       # Доля племенного поголовья (0-1)
    "previous_subsidies_count":     3.0,        # Предыдущие субсидии (0-15)
    "debt_load_ratio":              1.2,        # Долговая нагрузка (0-5)
    "log_amount":                   15.5,       # Логарифм суммы заявки
    "livestock_count":              50.0,       # Количество голов
    "direction_code":               0.0,        # 0=КРС, 1=овцы, 2=кони, 3=птица
    "is_pedigree":                  1.0,        # 1=племенное, 0=товарное
    "is_producer":                  0.0,        # 1=производители, 0=нет
    "hour_submitted":               12.0,       # Час подачи (0-23)
    "month_submitted":              3.0,        # Месяц подачи (1-12)
    "region_encoded":               0.0,        # Код региона (0-13)
}

# Создаём DataFrame
X = pd.DataFrame([applicant], columns=ML_FEATURES)

print("Данные заявителя:")
for k, v in applicant.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 3. Предсказание (ML Score)
# ============================================================

X_scaled = scaler.transform(X)
raw_score = float(model.predict(X_scaled)[0])
score = float(np.clip(raw_score, 1, 100))

if score >= 80:
    zone = "GREEN"
    label = "Строго рекомендовано"
elif score >= 50:
    zone = "YELLOW"
    label = "Требует рассмотрения"
else:
    zone = "RED"
    label = "Не рекомендовано"

print("=" * 50)
print(f"  ML SCORE: {score:.1f} / 100")
print(f"  Зона:     {zone}")
print(f"  Вердикт:  {label}")
print("=" * 50)

In [ ]:
# ============================================================
# 4. SHAP объяснение
# ============================================================
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_scaled)

if shap_values.ndim == 2:
    shap_values = shap_values[0]

FEATURE_LABELS = {
    "gross_output_growth_yoy":    "Рост валовой продукции",
    "land_to_livestock_ratio":    "Обеспеченность пастбищами",
    "historical_survival_rate":   "Сохранность поголовья",
    "subsidy_dependence_index":   "Зависимость от субсидий",
    "veterinary_compliance":      "Ветеринарное соответствие",
    "years_in_operation":         "Стаж работы",
    "pedigree_ratio":             "Доля племенного поголовья",
    "previous_subsidies_count":   "Предыдущие субсидии",
    "debt_load_ratio":            "Долговая нагрузка",
    "log_amount":                 "Масштаб заявки",
    "livestock_count":            "Количество голов",
    "direction_code":             "Направление",
    "is_pedigree":                "Племенное направление",
    "is_producer":                "Производители",
    "hour_submitted":             "Час подачи",
    "month_submitted":            "Месяц подачи",
    "region_encoded":             "Регион",
}

print("\nSHAP — вклад каждой фичи в балл:")
print(f"{'Фича':<35} {'Значение':>10} {'Вклад':>10}")
print("-" * 55)

factors = []
for name, shap_val in zip(ML_FEATURES, shap_values):
    raw_val = applicant[name]
    label = FEATURE_LABELS.get(name, name)
    sign = "+" if shap_val > 0 else ""
    print(f"{label:<35} {raw_val:>10.4f} {sign}{shap_val:>8.2f}")
    factors.append({"feature": label, "value": raw_val, "shap": shap_val})

# Топ-3 положительных и отрицательных
factors.sort(key=lambda x: abs(x["shap"]), reverse=True)

print("\n" + "=" * 50)
print("  ТОП влияния на балл:")
print("=" * 50)

pos = [f for f in factors if f["shap"] > 0][:3]
neg = [f for f in factors if f["shap"] < 0][:3]

if pos:
    print("\n  Повышают балл:")
    for f in pos:
        print(f"    +{f['shap']:.2f}: {f['feature']} = {f['value']:.2f}")

if neg:
    print("\n  Снижают балл:")
    for f in neg:
        print(f"    {f['shap']:.2f}: {f['feature']} = {f['value']:.2f}")